# W13C2 Lab: Memory, Reflection and Knowing If It Worked

Run every cell from the top. **Everything already works.**

Surface level by design: the aim is to recognise these three ideas and
know what they cost, not to implement any of them properly.

Today you will:

1. Give an agent memory and see what it can suddenly do.
2. Let it fail, reflect, and retry, then measure whether reflection helped.
3. Score a set of agents and see why one number is not enough.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import pandas as pd
import matplotlib.pyplot as plt

TASKS = [
    {"task": "add 17 and 25", "answer": 42},
    {"task": "multiply 6 by 7", "answer": 42},
    {"task": "subtract 8 from 50", "answer": 42},
    {"task": "divide 84 by 2", "answer": 42},
    {"task": "add 40 and 2", "answer": 42},
]
print(f"{len(TASKS)} tasks, all with the answer 42")

## Part 1. Memory: short term and long term

Short-term memory is what happened in this conversation. Long-term memory
is what the agent carries between conversations. They fail differently.

In [ ]:
# GIVEN. The same agent, with and without memory.
class Agent:
    def __init__(self, use_memory=False):
        self.use_memory = use_memory
        self.short_term = []      # this conversation
        self.long_term = {}       # kept between conversations

    def do(self, task):
        self.short_term.append(task)
        if self.use_memory and task in self.long_term:
            return self.long_term[task], "recalled"
        result = self._work(task)
        if self.use_memory:
            self.long_term[task] = result
        return result, "computed"

    def _work(self, task):
        words = task.split()
        a, b = int(words[1]), int(words[-1])
        if task.startswith("add"): return a + b
        if task.startswith("multiply"): return a * b
        if task.startswith("subtract"): return b - a
        if task.startswith("divide"): return a // b
        return None

plain, remembering = Agent(False), Agent(True)
for agent, label in ((plain, "no memory"), (remembering, "with memory")):
    print(f"--- {label} ---")
    for t in ["add 17 and 25", "multiply 6 by 7", "add 17 and 25"]:
        value, how = agent.do(t)
        print(f"   {t:<18} -> {value}  ({how})")
    print(f"   short-term holds {len(agent.short_term)} items, "
          f"long-term holds {len(agent.long_term)}")
    print()

In [ ]:
# ================== YOUR TURN 1 ==================
# Memory is not free. Ask the memory agent 200 different tasks and look
# at what its long-term store does.
#
# Expected: the store grows without limit, one entry per distinct task. That is
#           the real engineering problem behind agent memory: not how to store
#           things, but what to forget, and how to find the right item again once
#           there are thousands.
# ===============================================
N_TASKS = 200          # <-- try 20, then 2000

big = Agent(use_memory=True)
for i in range(N_TASKS):
    big.do(f"add {i} and {i}")

print(f"tasks seen         : {N_TASKS}")
print(f"long-term entries  : {len(big.long_term)}")
print(f"short-term entries : {len(big.short_term)}")
print()
print("Nothing here ever forgets, and lookup is exact-match only.")
print("A real agent needs both a forgetting policy and fuzzy retrieval.")

## Part 2. Reflection: fail, think about it, try again

Reflexion's idea in one sentence: when an attempt fails, write down WHY in
plain language, put that note in the next prompt, and try again. No
weights are updated.

In [ ]:
# GIVEN. An agent that gets subtraction backwards, until it reflects.
def attempt(task, lessons):
    words = task.split()
    a, b = int(words[1]), int(words[-1])
    if task.startswith("subtract"):
        # The bug: it subtracts the wrong way round, unless it has been told.
        if "subtract: take the first number FROM the second" in lessons:
            return b - a
        return a - b
    if task.startswith("add"): return a + b
    if task.startswith("multiply"): return a * b
    if task.startswith("divide"): return a // b

def reflect(task, got, want):
    return f"subtract: take the first number FROM the second"

def solve(task, want, use_reflection, max_tries=2):
    lessons, tries = [], 0
    for tries in range(1, max_tries + 1):
        got = attempt(task, lessons)
        if got == want:
            return got, tries
        if not use_reflection:
            break
        lessons.append(reflect(task, got, want))
    return got, tries

for use in (False, True):
    got, tries = solve("subtract 8 from 50", 42, use_reflection=use)
    print(f"   reflection={use!s:<5} answer {got:<4} after {tries} attempt(s)")

In [ ]:
# ================== YOUR TURN 2 ==================
# Measure reflection over the whole task set instead of one example.
#
# Expected: reflection fixes the subtraction task and changes nothing else, so
#           accuracy goes from 0.80 to 1.00 while the number of model calls goes
#           up. Reflection buys accuracy with latency and money, and only on tasks
#           where the failure is diagnosable from the output.
# ===============================================
USE_REFLECTION = False          # <-- set to True

correct = calls = 0
for item in TASKS:
    got, tries = solve(item["task"], item["answer"], use_reflection=USE_REFLECTION)
    correct += got == item["answer"]
    calls += tries

print(f"reflection: {USE_REFLECTION}")
print(f"   accuracy    : {correct / len(TASKS):.2f}")
print(f"   model calls : {calls}  (one per attempt)")

## Part 3. Did it actually work?

The golden rule of agent evaluation: score the OUTCOME, not the trace. An
agent that produces a beautiful chain of reasoning and the wrong answer
has failed.

In [ ]:
# GIVEN. Four agents, scored three ways.
AGENTS = {
    "correct but terse":   {"right": 5, "steps": 5,  "words": 40},
    "correct and verbose": {"right": 5, "steps": 22, "words": 900},
    "wrong but eloquent":  {"right": 1, "steps": 18, "words": 850},
    "refuses everything":  {"right": 0, "steps": 1,  "words": 20},
}
rows = []
for name, a in AGENTS.items():
    rows.append({"agent": name,
                 "accuracy": a["right"] / 5,
                 "steps used": a["steps"],
                 "looks thorough": round(min(a["words"] / 900, 1.0), 2)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, col in zip(axes, ["accuracy", "steps used", "looks thorough"]):
    ax.bar(table["agent"], table[col], color="#7C2529")
    ax.set_title(col); ax.tick_params(axis="x", rotation=40, labelsize=7)
plt.tight_layout(); plt.show()
print("'wrong but eloquent' looks best on the third chart and solved one task.")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   The store grows one entry per distinct task, forever, and lookup is exact
#   string match. Real agent memory needs a forgetting policy and fuzzy
#   retrieval, which in practice means embedding the memories and searching them
#   the same way you searched documents in Week 12.
#
# YOUR TURN 2
#   Accuracy goes 0.80 -> 1.00 and calls go 5 -> 6. Reflection only works when
#   the failure is visible in the output and describable in words. It cannot fix
#   a task the agent has no way to check.
#
# PART 3, nothing to edit
#   Score outcomes. Step counts and eloquence are diagnostics, not scores, and
#   an agent optimised against them will produce longer traces rather than
#   better answers, which is the same reward hacking you saw in Week 9.